# Mixed traffic capacity pilot
Run on a T4. The default p95 target is 500 ms. Edit --target-ms in the measurement cell to change it. Each rate uses two 60-request traces per policy. A passing trace completes every request and meets the latency target.

In [ ]:
import os
import shutil
import subprocess
import tempfile

smi = shutil.which("nvidia-smi")
if not smi:
    raise RuntimeError("No NVIDIA GPU is available. In Colab, select Runtime → Change runtime type → T4 GPU (or another available NVIDIA GPU), reconnect, and rerun from the first cell.")
subprocess.run([smi, "-L"], check=True)

%pip -q install uv
os.environ["VOICEFORGE_PHASE8_RESULTS"] = tempfile.mkdtemp(prefix="voiceforge-phase8-", dir="/content")

In [ ]:
%%bash
set -e
if ! command -v nvidia-smi >/dev/null || ! nvidia-smi -L; then
  echo "No NVIDIA GPU is available. In Colab, select Runtime → Change runtime type → T4 GPU (or another available NVIDIA GPU), reconnect, and rerun from the first cell." >&2
  exit 1
fi
if ! command -v espeak-ng >/dev/null; then
  apt-get -qq update
  apt-get -qq install -y espeak-ng
fi
repo=/content/voiceforge-capacity
if [[ ! -d "$repo/.git" ]]; then
  git clone --branch main https://github.com/dne-labs/Voice-inference.git "$repo"
fi
git -C "$repo" checkout --detach 28189610510bfda9b30e28819aeec8e866c03ef8
cd "$repo/services/inference"
uv run --locked --python 3.12 \
  --index https://download.pytorch.org/whl/cu126 \
  --with 'torch==2.8.0+cu126' --with 'kokoro==0.9.4' \
  --with 'en-core-web-sm @ https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl' \
  voiceforge-synthesize --device cuda --output /tmp/voiceforge-phase8-warmup.wav

In [ ]:
%%bash
set -e
if ! command -v nvidia-smi >/dev/null || ! nvidia-smi -L; then
  echo "No NVIDIA GPU is available. In Colab, select Runtime → Change runtime type → T4 GPU (or another available NVIDIA GPU), reconnect, and rerun from the first cell." >&2
  exit 1
fi
cd /content/voiceforge-capacity/services/inference
HF_HUB_OFFLINE=1 OMP_NUM_THREADS=2 uv run --locked --python 3.12 \
  --index https://download.pytorch.org/whl/cu126 \
  --with 'torch==2.8.0+cu126' --with 'kokoro==0.9.4' \
  --with 'en-core-web-sm @ https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl' \
  python ../../experiments/06_adaptive_chunking/capacity.py --target-ms 500 --output "$VOICEFORGE_PHASE8_RESULTS/capacity.json"

In [ ]:
from pathlib import Path
import json, os
from google.colab import files

path = Path(os.environ["VOICEFORGE_PHASE8_RESULTS"]) / "capacity.json"
print(json.loads(path.read_text())["highest_passing_tested_rpm"])
files.download(str(path))